# Exercices — Preparation Morgan Stanley Data Engineer

Ce notebook regroupe des exercices pratiques sur les 4 axes de revision :
1. **Python / Pandas** pour le data engineering
2. **SQL Avance** (fenetrage, CTE, optimisation)
3. **Snowflake** (architecture, chargement, streams/tasks)
4. **Data Modeling & ETL** (Kimball, SCD, pipelines)

---

## Partie 1 — Python / Pandas

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

### Exercice 1.1 — Creation et exploration d'un DataFrame

Creer un DataFrame `trades` avec les colonnes suivantes :
- `trade_id` : [101, 102, 103, 104, 105, 106, 107, 108]
- `trader` : ['Alice', 'Bob', 'Alice', 'Charlie', 'Bob', 'Alice', 'Charlie', 'Bob']
- `instrument` : ['AAPL', 'GOOG', 'AAPL', 'MSFT', 'GOOG', 'TSLA', 'MSFT', 'AAPL']
- `quantity` : [100, 50, -30, 200, 75, 150, -50, 80]
- `price` : [150.5, 2800.0, 151.0, 310.0, 2750.0, 900.0, 312.0, 149.0]
- `date` : 8 dates consecutives a partir du 2024-01-15

Afficher : shape, dtypes, et les 3 premieres lignes.

In [7]:
# Votre code ici
trades = pd.DataFrame({
    'trade_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'trader': ['Alice', 'Bob', 'Alice', 'Charlie', 'Bob', 'Alice', 'Charlie', 'Bob'],
    'instrument': ['AAPL', 'GOOG', 'AAPL', 'MSFT', 'GOOG', 'TSLA', 'MSFT', 'AAPL'],
    'quantity': [100, 50, -30, 200, 75, 150, -50, 80],
    'price': [150.5, 2800.0, 151.0, 310.0, 2750.0, 900.0, 312.0, 149.0],
    'date': pd.date_range(start='2024-01-15', periods=8, freq='D')
})
df

,trade_id,trader,instrument,quantity,price,date
0,101,Alice,AAPL,100,150.5,2024-01-15
1,102,Bob,GOOG,50,2800.0,2024-01-16
2,103,Alice,AAPL,-30,151.0,2024-01-17
3,104,Charlie,MSFT,200,310.0,2024-01-18
4,105,Bob,GOOG,75,2750.0,2024-01-19
5,106,Alice,TSLA,150,900.0,2024-01-20
6,107,Charlie,MSFT,-50,312.0,2024-01-21
7,108,Bob,AAPL,80,149.0,2024-01-22


### Exercice 1.2 — Filtrage et colonnes calculees

A partir du DataFrame `trades` :
1. Filtrer les trades d'achat uniquement (`quantity > 0`)
2. Ajouter une colonne `montant` = `quantity * price`
3. Ajouter une colonne `type_trade` : 'ACHAT' si quantity > 0, 'VENTE' sinon
4. Filtrer les trades ou `montant > 50000` et l'instrument est 'GOOG' ou 'AAPL'

In [19]:
trades['montant'] = trades['quantity'] * trades['price']
trades

,trade_id,trader,instrument,quantity,price,date,montant,type_trade
0,101,Alice,AAPL,100,150.5,2024-01-15,15050.0,Achat
1,102,Bob,GOOG,50,2800.0,2024-01-16,140000.0,Achat
2,103,Alice,AAPL,-30,151.0,2024-01-17,-4530.0,Vente
3,104,Charlie,MSFT,200,310.0,2024-01-18,62000.0,Achat
4,105,Bob,GOOG,75,2750.0,2024-01-19,206250.0,Achat
5,106,Alice,TSLA,150,900.0,2024-01-20,135000.0,Achat
6,107,Charlie,MSFT,-50,312.0,2024-01-21,-15600.0,Vente
7,108,Bob,AAPL,80,149.0,2024-01-22,11920.0,Achat


In [20]:
df_qte_positive = trades[trades['quantity'] > 0]
df_qte_positive


,trade_id,trader,instrument,quantity,price,date,montant,type_trade
0,101,Alice,AAPL,100,150.5,2024-01-15,15050.0,Achat
1,102,Bob,GOOG,50,2800.0,2024-01-16,140000.0,Achat
3,104,Charlie,MSFT,200,310.0,2024-01-18,62000.0,Achat
4,105,Bob,GOOG,75,2750.0,2024-01-19,206250.0,Achat
5,106,Alice,TSLA,150,900.0,2024-01-20,135000.0,Achat
7,108,Bob,AAPL,80,149.0,2024-01-22,11920.0,Achat


In [21]:
trades['type_trade'] = np.where(trades['quantity'] > 0, 'Achat', 'Vente')
trades


,trade_id,trader,instrument,quantity,price,date,montant,type_trade
0,101,Alice,AAPL,100,150.5,2024-01-15,15050.0,Achat
1,102,Bob,GOOG,50,2800.0,2024-01-16,140000.0,Achat
2,103,Alice,AAPL,-30,151.0,2024-01-17,-4530.0,Vente
3,104,Charlie,MSFT,200,310.0,2024-01-18,62000.0,Achat
4,105,Bob,GOOG,75,2750.0,2024-01-19,206250.0,Achat
5,106,Alice,TSLA,150,900.0,2024-01-20,135000.0,Achat
6,107,Charlie,MSFT,-50,312.0,2024-01-21,-15600.0,Vente
7,108,Bob,AAPL,80,149.0,2024-01-22,11920.0,Achat


In [22]:
# Using apply to create the conditional column
#trades['type_trade'] = trades['quantity'].apply(lambda x: 'Achat' if x > 0 else 'Vente')

# # Equivalent SQL CASE WHEN (for reference)
# sql_type_trade = """
# SELECT *,
#     CASE WHEN quantity > 0 THEN 'Achat' ELSE 'Vente' END AS type_trade
# FROM trades
# """

In [23]:
trades

,trade_id,trader,instrument,quantity,price,date,montant,type_trade
0,101,Alice,AAPL,100,150.5,2024-01-15,15050.0,Achat
1,102,Bob,GOOG,50,2800.0,2024-01-16,140000.0,Achat
2,103,Alice,AAPL,-30,151.0,2024-01-17,-4530.0,Vente
3,104,Charlie,MSFT,200,310.0,2024-01-18,62000.0,Achat
4,105,Bob,GOOG,75,2750.0,2024-01-19,206250.0,Achat
5,106,Alice,TSLA,150,900.0,2024-01-20,135000.0,Achat
6,107,Charlie,MSFT,-50,312.0,2024-01-21,-15600.0,Vente
7,108,Bob,AAPL,80,149.0,2024-01-22,11920.0,Achat


### Exercice 1.3 — Groupby et agregation

A partir du DataFrame `trades` (avec la colonne `montant`) :
1. Calculer par trader : nombre de trades, montant total, montant moyen
2. Calculer par instrument : quantite nette (somme des quantity), prix moyen pondere par quantite
3. Utiliser `transform` pour ajouter une colonne `montant_moyen_trader` (moyenne du montant par trader, reportee sur chaque ligne)

In [36]:
df_by_traders = trades.groupby('trader').agg(
    nombre_de_trades=('trade_id', 'count'),
    montant_total=('montant', 'sum'),
    montant_moyen=('montant', 'mean')
).reset_index()
df_by_traders

,trader,nombre_de_trades,montant_total,montant_moyen
0,Alice,3,145520.0,48506.666667
1,Bob,3,358170.0,119390.000000
2,Charlie,2,46400.0,23200.000000


### Exercice 1.4 — Jointures (merge)

Creer un DataFrame `traders_info` :
```python
traders_info = pd.DataFrame({
    'trader': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'desk': ['Equities', 'Equities', 'Fixed Income', 'FX'],
    'seniority': ['VP', 'Associate', 'VP', 'Director']
})
```

1. Faire un **left join** de `trades` avec `traders_info` sur `trader`
2. Verifier s'il y a des traders dans `trades` absents de `traders_info` (et inversement)
3. Calculer le montant total par `desk`

In [35]:
traders_in_trades = set(trades['trader'])
traders_in_info = set(traders_info['trader'])

absent_from_info = traders_in_trades - traders_in_info
absent_from_trades = traders_in_info - traders_in_trades

print("Traders présents dans trades mais absents de traders_info :", absent_from_info)
print("Traders présents dans traders_info mais absents de trades :", absent_from_trades)

Traders présents dans trades mais absents de traders_info : set()
Traders présents dans traders_info mais absents de trades : {'Diana'}


In [30]:
# Votre code ici
traders_info = pd.DataFrame({
    'trader': ['Alice', 'Bob', 'Charlie', 'Diana'],
    'desk': ['Equities', 'Equities', 'Fixed Income', 'FX'],
    'seniority': ['VP', 'Associate', 'VP', 'Director']
})
traders_with_infos = pd.merge(trades,traders_info,on='trader', how='left')
traders_with_infos

,trade_id,trader,instrument,quantity,price,date,montant,type_trade,desk,seniority
0,101,Alice,AAPL,100,150.5,2024-01-15,15050.0,Achat,Equities,VP
1,102,Bob,GOOG,50,2800.0,2024-01-16,140000.0,Achat,Equities,Associate
2,103,Alice,AAPL,-30,151.0,2024-01-17,-4530.0,Vente,Equities,VP
3,104,Charlie,MSFT,200,310.0,2024-01-18,62000.0,Achat,Fixed Income,VP
4,105,Bob,GOOG,75,2750.0,2024-01-19,206250.0,Achat,Equities,Associate
5,106,Alice,TSLA,150,900.0,2024-01-20,135000.0,Achat,Equities,VP
6,107,Charlie,MSFT,-50,312.0,2024-01-21,-15600.0,Vente,Fixed Income,VP
7,108,Bob,AAPL,80,149.0,2024-01-22,11920.0,Achat,Equities,Associate


In [34]:
mont_by_desk = traders_with_infos.groupby('desk')['montant'].sum().reset_index()
mont_by_desk

,desk,montant
0,Equities,503690.0
1,Fixed Income,46400.0


### Exercice 1.5 — Pivot table et crosstab

1. Creer un **pivot_table** montrant le montant total par `trader` (lignes) et `instrument` (colonnes)
2. Creer un **crosstab** montrant le nombre de trades par `trader` et `type_trade`

In [37]:
# Votre code ici
pivot_table = trades.pivot_table(index='trader', columns='instrument', values='montant', aggfunc='sum', fill_value=0)
pivot_table

instrument,AAPL,GOOG,MSFT,TSLA
trader,,,,
Alice,10520.0,0.0,0.0,135000.0
Bob,11920.0,346250.0,0.0,0.0
Charlie,0.0,0.0,46400.0,0.0


In [38]:
cross_table = pd.crosstab(trades['trader'], trades['instrument'], values=trades['trade_id'], aggfunc='count', dropna=False).fillna(0)
cross_table                                                                                                                              

instrument,AAPL,GOOG,MSFT,TSLA
trader,,,,
Alice,2.0,0.0,0.0,1.0
Bob,1.0,2.0,0.0,0.0
Charlie,0.0,0.0,2.0,0.0


### Exercice 1.6 — Gestion des valeurs manquantes et doublons

Creer le DataFrame suivant :
```python
df_dirty = pd.DataFrame({
    'id': [1, 2, 3, 3, 4, 5],
    'valeur': [10.0, np.nan, 30.0, 30.0, np.nan, 50.0],
    'categorie': ['A', 'B', 'A', 'A', 'B', None]
})
```

1. Identifier les lignes avec des valeurs manquantes
2. Remplir les NaN de `valeur` par la mediane
3. Remplir les None de `categorie` par 'INCONNU'
4. Supprimer les doublons sur `id`

In [39]:
df_dirty = pd.DataFrame({
    'id': [1, 2, 3, 3, 4, 5],
    'valeur': [10.0, np.nan, 30.0, 30.0, np.nan, 50.0],
    'categorie': ['A', 'B', 'A', 'A', 'B', None]
})
df_dirty



,id,valeur,categorie
0,1,10.0,A
1,2,NaN,B
2,3,30.0,A
3,3,30.0,A
4,4,NaN,B
5,5,50.0,NaN


In [40]:
df_dirty.notna()


,id,valeur,categorie
0,True,True,True
1,True,False,True
2,True,True,True
3,True,True,True
4,True,False,True
5,True,True,False


In [43]:
# identifier les lignes avec des valeurs manquantes
missing_values = df_dirty.isna()
missing_values

,id,valeur,categorie
0,False,False,False
1,False,True,False
2,False,False,False
3,False,False,False
4,False,True,False
5,False,False,True


### Exercice 1.7 — Series temporelles et resampling

Creer un DataFrame de prix journaliers sur 60 jours :
```python
np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=60, freq='B')  # jours ouvres
df_prix = pd.DataFrame({
    'date': dates,
    'prix': 100 + np.random.randn(60).cumsum()
}).set_index('date')
```

1. Calculer la **moyenne mobile** sur 5 jours (`rolling`)
2. Calculer le **rendement journalier** (pct_change)
3. **Resampler** en frequence hebdomadaire : prix moyen, min et max par semaine
4. Trouver la semaine avec la plus grande amplitude (max - min)

In [47]:
np.random.seed(42)
dates = pd.date_range('2024-01-01', periods=60, freq='B')  # jours ouvres
df_prix = pd.DataFrame({
    'date': dates,
    'prix': 100 + np.random.randn(60).cumsum()
}).set_index('date')
df_prix.head(5)

,prix
date,
2024-01-01,100.496714
2024-01-02,100.358450
2024-01-03,101.006138
2024-01-04,102.529168
2024-01-05,102.295015


In [50]:
df_prix['moyenne_mobile_5'] = df_prix['prix'].rolling(window=5).mean().fillna(0)
df_prix.head(10)

,prix,moyenne_mobile_5
date,,
2024-01-01,100.496714,0.000000
2024-01-02,100.358450,0.000000
2024-01-03,101.006138,0.000000
2024-01-04,102.529168,0.000000
2024-01-05,102.295015,101.337097
2024-01-08,102.060878,101.649930
2024-01-09,103.640091,102.306258
2024-01-10,104.407525,102.986535
2024-01-11,103.938051,103.268312


In [53]:
df_prix['rendement_journalier_pct'] = df_prix['prix'].pct_change() * 100
df_prix['rendement_journalier_pct'] = df_prix['rendement_journalier_pct'].fillna(0)
df_prix.head(10)

,prix,moyenne_mobile_5,rendement_journalier_pct
date,,,
2024-01-01,100.496714,0.000000,0.000000
2024-01-02,100.358450,0.000000,-0.137581
2024-01-03,101.006138,0.000000,0.645375
2024-01-04,102.529168,0.000000,1.507859
2024-01-05,102.295015,101.337097,-0.228377
2024-01-08,102.060878,101.649930,-0.228884
2024-01-09,103.640091,102.306258,1.547324
2024-01-10,104.407525,102.986535,0.740481
2024-01-11,103.938051,103.268312,-0.449656


In [54]:
df_semaine = df_prix.resample('W').agg(
    prix_moyen=('prix', 'mean'),
    prix_min=('prix', 'min'),
    prix_max=('prix', 'max')
)
df_semaine.head(5)

,prix_moyen,prix_min,prix_max
date,,,
2024-01-07,101.337097,100.358450,102.529168
2024-01-14,103.705431,102.060878,104.480611
2024-01-21,102.679491,100.155228,104.017193
2024-01-28,98.325553,96.574029,99.592940
2024-02-04,97.220798,95.912299,98.039678


In [60]:
df_amplitude = df_prix.resample('W').agg(
    prix_max=('prix', 'max'),
    prix_min=('prix', 'min'),
    amplitude=('prix', lambda x: x.max() - x.min())
)
meilleure_semaine = df_amplitude['amplitude'].idxmax()
print(meilleure_semaine)
print(" ")
list_semaines = df_amplitude.sort_values('amplitude', ascending=False)
list_semaines['amplitude']

2024-01-21 00:00:00
 


date
2024-01-21    3.861966
2024-02-25    3.287856
2024-01-28    3.018912
2024-01-14    2.419733
2024-01-07    2.170718
2024-02-04    2.127379
2024-03-03    1.895274
2024-02-18    1.852278
2024-03-10    1.763040
2024-02-11    1.667628
2024-03-17    1.642676
2024-03-24    1.306809
Name: amplitude, dtype: float64

### Exercice 1.8 — Lecture/ecriture de fichiers et performance

1. Generer un DataFrame de 100 000 lignes avec : `id`, `valeur` (float aleatoire), `categorie` (choix parmi ['A','B','C','D'])
2. Sauvegarder en CSV puis en Parquet
3. Comparer la **taille des fichiers** et le **temps de lecture** (`%%timeit` ou `time`)
4. Lire le fichier Parquet en ne chargeant que les colonnes `id` et `valeur`

In [ ]:
# Votre code ici


---
## Partie 2 — SQL Avance

> Pour ces exercices, ecrivez les requetes SQL dans des cellules Markdown ou en commentaire Python.
> Si vous avez SQLite ou DuckDB, vous pouvez les executer directement.

### Exercice 2.1 — Window Functions : classement

Table `employes(id, nom, departement, salaire, date_embauche)`

Ecrire une requete qui retourne pour chaque employe :
- Son `nom`, `departement`, `salaire`
- Son **rang** dans son departement par salaire decroissant (`RANK`)
- Son **rang dense** (`DENSE_RANK`)
- Son **numero de ligne** (`ROW_NUMBER`)
- Le **quartile** dans lequel il se trouve (`NTILE(4)`) sur l'ensemble de la table

In [ ]:
# Ecrivez votre requete SQL ici (en commentaire ou string)
sql_2_1 = """

"""

### Exercice 2.2 — Window Functions : agregation et decalage

Table `ventes(id, produit, region, montant, date_vente)`

Ecrire une requete qui retourne :
- Chaque vente avec son `montant`
- Le **cumul** du montant par region, ordonne par date
- La **moyenne mobile** sur 3 lignes (courante + 2 precedentes) par region
- Le montant de la **vente precedente** dans la meme region (`LAG`)
- La **variation** par rapport a la vente precedente

In [ ]:
sql_2_2 = """

"""

### Exercice 2.3 — CTE et requetes recursives

Table `employes(id, nom, manager_id, departement, salaire)`

1. Ecrire une **CTE recursive** qui retourne la hierarchie complete depuis le PDG (manager_id IS NULL) avec le niveau hierarchique
2. Ecrire une CTE qui calcule le salaire total par departement, puis retourne les departements dont le salaire total depasse la moyenne des salaires totaux de tous les departements

In [ ]:
sql_2_3 = """

"""

### Exercice 2.4 — Top-N par groupe

Table `transactions(id, client_id, montant, date_transaction, type_transaction)`

Ecrire une requete qui retourne les **3 plus grosses transactions** par client, en incluant :
- le rang de la transaction
- le pourcentage que cette transaction represente par rapport au total du client

In [ ]:
sql_2_4 = """

"""

### Exercice 2.5 — PIVOT / UNPIVOT et agregation conditionnelle

Table `ventes_mensuelles(produit, mois, montant)`

1. Ecrire une requete pour **pivoter** les donnees : une ligne par produit, une colonne par mois (Jan, Fev, Mar)
2. Ecrire la requete inverse (UNPIVOT) si les mois sont en colonnes
3. Ecrire une requete avec `CASE WHEN` pour obtenir le meme resultat que le pivot

In [ ]:
sql_2_5 = """

"""

### Exercice 2.6 — Gaps and Islands

Table `connexions(user_id, date_connexion DATE)` (une ligne par jour de connexion)

Ecrire une requete qui identifie les **series consecutives** de connexion pour chaque utilisateur :
- date de debut de la serie
- date de fin
- nombre de jours consecutifs

*Indice : utiliser ROW_NUMBER et la difference avec la date pour identifier les groupes.*

In [ ]:
sql_2_6 = """

"""

---
## Partie 3 — Snowflake

### Exercice 3.1 — Architecture (QCM)

Repondre aux questions suivantes :

1. Quelles sont les 3 couches de l'architecture Snowflake ? Decrivez le role de chacune.

2. Vrai ou Faux :
   - a) Deux Virtual Warehouses peuvent partager leur cache local
   - b) Un warehouse XL coute 16x plus qu'un XS
   - c) Les micro-partitions font entre 50 et 500 MB compressees
   - d) Time Travel permet de remonter jusqu'a 90 jours sur l'edition Enterprise

3. Quelle est la difference entre `COPY INTO` et `SNOWPIPE` pour le chargement de donnees ?

*Vos reponses ici :*



### Exercice 3.2 — Chargement de donnees

Ecrire les commandes Snowflake pour :
1. Creer un **stage externe** pointant vers un bucket S3
2. Creer un **file format** CSV avec delimiteur `;`, en-tete, et gestion des NULL
3. Charger les donnees avec `COPY INTO` en ignorant les erreurs (max 10 erreurs tolerees)
4. Verifier le resultat du chargement avec `VALIDATE`

In [ ]:
sql_3_2 = """

"""

### Exercice 3.3 — Streams et Tasks

Ecrire les commandes pour mettre en place un pipeline CDC :
1. Creer une table `raw_trades` et une table `processed_trades`
2. Creer un **stream** sur `raw_trades`
3. Creer une **task** planifiee toutes les 5 minutes qui :
   - Verifie si le stream contient des donnees (`SYSTEM$STREAM_HAS_DATA`)
   - Insere les nouvelles lignes dans `processed_trades` avec une colonne `processed_at`
4. Activer la task

In [ ]:
sql_3_3 = """

"""

### Exercice 3.4 — Time Travel et clonage

1. Ecrire la requete pour consulter l'etat d'une table `positions` il y a exactement 30 minutes
2. Ecrire la commande pour restaurer une table `positions_backup` supprimee accidentellement
3. Creer un **clone zero-copy** de la table `trades` pour un environnement de dev
4. Expliquer pourquoi le clone est "zero-copy" et dans quels cas il commence a consommer du stockage

In [ ]:
sql_3_4 = """

"""

### Exercice 3.5 — Optimisation de requetes

Expliquer comment optimiser chacun des scenarios suivants :

1. Une requete `SELECT *` sur une table de 500M lignes filtree par `date_trade` est lente
2. Un Virtual Warehouse XL est utilise 2 heures par jour mais reste actif 24h
3. Plusieurs equipes executent des requetes concurrentes sur le meme warehouse et subissent des files d'attente
4. Une table contient 80% de colonnes VARCHAR rarement utilisees et les scans sont lents

*Vos reponses ici :*



---
## Partie 4 — Data Modeling & ETL

### Exercice 4.1 — Schema en etoile

Concevoir un schema en etoile pour un systeme de **suivi des transactions financieres** :

1. Identifier la **table de faits** et ses mesures
2. Identifier au moins **4 dimensions** avec leurs attributs principaux
3. Ecrire les `CREATE TABLE` SQL correspondants
4. Identifier une dimension qui pourrait etre **role-playing** et une **junk dimension**

In [ ]:
sql_4_1 = """

"""

### Exercice 4.2 — SCD (Slowly Changing Dimensions)

Un client change d'adresse. Ecrire les requetes SQL pour gerer ce changement selon :

1. **SCD Type 1** : ecrasement (pas d'historique)
2. **SCD Type 2** : historisation (nouvelle ligne avec dates de validite)
3. **SCD Type 3** : colonne precedente (ancien + nouveau)

Table de depart :
```sql
dim_client(client_key, client_id, nom, adresse, ville, date_debut, date_fin, est_courant)
```

Le client `client_id = 42` demenage de 'Paris' a 'Lyon' le 2024-03-15.

In [ ]:
sql_4_2 = """

"""

### Exercice 4.3 — ETL vs ELT

Repondre aux questions :

1. Quelle est la difference fondamentale entre ETL et ELT ?
2. Pourquoi ELT est-il plus adapte a un environnement Snowflake ?
3. Decrire un pipeline ELT en 4 etapes pour charger des fichiers CSV d'un bucket S3 vers un modele etoile dans Snowflake
4. Quels mecanismes garantissent l'**idempotence** du pipeline ?

*Vos reponses ici :*



### Exercice 4.4 — MERGE / Upsert

Ecrire une requete `MERGE` qui :
- Fusionne une table `staging_positions` dans `dim_positions`
- Met a jour les lignes existantes si le `montant` a change
- Insere les nouvelles lignes
- Marque comme inactives les lignes de `dim_positions` absentes du staging

Tables :
```sql
staging_positions(position_id, instrument, montant, date_maj)
dim_positions(position_key, position_id, instrument, montant, est_actif, date_debut, date_fin)
```

In [ ]:
sql_4_4 = """

"""

### Exercice 4.5 — Qualite des donnees

Ecrire en Python (Pandas) des fonctions de validation pour un DataFrame `df_trades` :

1. `check_nulls(df, colonnes)` : retourne les colonnes ayant des NaN et leur pourcentage
2. `check_duplicates(df, cles)` : retourne les doublons sur les colonnes cles
3. `check_range(df, colonne, min_val, max_val)` : retourne les lignes hors bornes
4. `check_referential_integrity(df_fact, df_dim, fk, pk)` : verifie que toutes les FK existent dans la dimension

In [ ]:
# Votre code ici

def check_nulls(df, colonnes):
    pass

def check_duplicates(df, cles):
    pass

def check_range(df, colonne, min_val, max_val):
    pass

def check_referential_integrity(df_fact, df_dim, fk, pk):
    pass


---
## Partie Bonus — Exercices mixtes type HackerRank

### Bonus 1 — Analyse de portefeuille (Python)

Donnees :
```python
portfolio = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=252, freq='B'),
    'AAPL': 100 + np.random.randn(252).cumsum() * 2,
    'GOOG': 150 + np.random.randn(252).cumsum() * 3,
    'MSFT': 80 + np.random.randn(252).cumsum() * 1.5
})
```

1. Calculer la **matrice de correlation** des rendements journaliers
2. Trouver le **drawdown maximum** pour chaque instrument (baisse max depuis un pic)
3. Calculer la **volatilite annualisee** (ecart-type des rendements * sqrt(252))
4. Identifier les jours ou au moins 2 instruments ont baisse de plus de 2%

In [ ]:
# Votre code ici


### Bonus 2 — Pipeline ETL complet (Python)

Simuler un mini pipeline ETL :

1. **Extract** : Generer 3 fichiers CSV simulant des sources differentes (clients, produits, transactions)
2. **Transform** :
   - Nettoyer les doublons et valeurs manquantes
   - Standardiser les formats de dates
   - Ajouter des cles de substitution (surrogate keys)
   - Creer une junk dimension a partir de colonnes booleennes
3. **Load** : Sauvegarder en Parquet partitionne par date

Utiliser des fonctions bien structurees et gerer les erreurs.

In [ ]:
# Votre code ici


### Bonus 3 — Requete SQL complexe

Tables :
- `accounts(account_id, client_id, account_type, open_date, status)`
- `transactions(txn_id, account_id, txn_date, amount, txn_type)`
- `clients(client_id, name, segment, region)`

Ecrire une seule requete SQL qui retourne pour chaque client :
- Son `name` et `segment`
- Le nombre de comptes actifs
- Le montant total des transactions des 30 derniers jours
- Le rang du client par montant total dans son segment
- La variation du montant mensuel par rapport au mois precedent (LAG)
- Seulement les clients dans le top 10 de leur segment

In [ ]:
sql_bonus_3 = """

"""

---
## Solutions

Les solutions sont disponibles dans les cellules ci-dessous. **Essayez de resoudre les exercices avant de regarder !**

In [ ]:
# === SOLUTION 1.1 ===
trades = pd.DataFrame({
    'trade_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'trader': ['Alice', 'Bob', 'Alice', 'Charlie', 'Bob', 'Alice', 'Charlie', 'Bob'],
    'instrument': ['AAPL', 'GOOG', 'AAPL', 'MSFT', 'GOOG', 'TSLA', 'MSFT', 'AAPL'],
    'quantity': [100, 50, -30, 200, 75, 150, -50, 80],
    'price': [150.5, 2800.0, 151.0, 310.0, 2750.0, 900.0, 312.0, 149.0],
    'date': pd.date_range('2024-01-15', periods=8, freq='D')
})
print(trades.shape)
print(trades.dtypes)
trades.head(3)

In [ ]:
# === SOLUTION 1.2 ===
# 1. Achats uniquement
achats = trades[trades['quantity'] > 0]

# 2. Montant
trades['montant'] = trades['quantity'] * trades['price']

# 3. Type de trade
trades['type_trade'] = np.where(trades['quantity'] > 0, 'ACHAT', 'VENTE')

# 4. Filtrage combine
filtre = trades[
    (trades['montant'] > 50000) &
    (trades['instrument'].isin(['GOOG', 'AAPL']))
]
print(filtre)

In [ ]:
# === SOLUTION 1.3 ===
# 1. Stats par trader
stats_trader = trades.groupby('trader').agg(
    nb_trades=('trade_id', 'count'),
    montant_total=('montant', 'sum'),
    montant_moyen=('montant', 'mean')
).reset_index()
print(stats_trader)

# 2. Stats par instrument
stats_instr = trades.groupby('instrument').agg(
    quantite_nette=('quantity', 'sum'),
).reset_index()
# Prix moyen pondere
trades_achat = trades[trades['quantity'] > 0]
prix_pondere = trades_achat.groupby('instrument').apply(
    lambda g: np.average(g['price'], weights=g['quantity'])
).reset_index(name='prix_moyen_pondere')
print(prix_pondere)

# 3. Transform
trades['montant_moyen_trader'] = trades.groupby('trader')['montant'].transform('mean')
trades[['trader', 'montant', 'montant_moyen_trader']].head()

In [ ]:
# === SOLUTION 2.1 ===
sql_2_1_solution = """
SELECT
    nom,
    departement,
    salaire,
    RANK() OVER (PARTITION BY departement ORDER BY salaire DESC) AS rang,
    DENSE_RANK() OVER (PARTITION BY departement ORDER BY salaire DESC) AS rang_dense,
    ROW_NUMBER() OVER (PARTITION BY departement ORDER BY salaire DESC) AS num_ligne,
    NTILE(4) OVER (ORDER BY salaire DESC) AS quartile
FROM employes
ORDER BY departement, salaire DESC;
"""
print(sql_2_1_solution)

In [ ]:
# === SOLUTION 2.2 ===
sql_2_2_solution = """
SELECT
    id,
    produit,
    region,
    montant,
    date_vente,
    SUM(montant) OVER (
        PARTITION BY region ORDER BY date_vente
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumul_montant,
    AVG(montant) OVER (
        PARTITION BY region ORDER BY date_vente
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS moyenne_mobile_3,
    LAG(montant, 1) OVER (
        PARTITION BY region ORDER BY date_vente
    ) AS montant_precedent,
    montant - LAG(montant, 1) OVER (
        PARTITION BY region ORDER BY date_vente
    ) AS variation
FROM ventes
ORDER BY region, date_vente;
"""
print(sql_2_2_solution)

In [ ]:
# === SOLUTION 2.6 (Gaps and Islands) ===
sql_2_6_solution = """
WITH numbered AS (
    SELECT
        user_id,
        date_connexion,
        date_connexion - INTERVAL '1 day' * ROW_NUMBER() OVER (
            PARTITION BY user_id ORDER BY date_connexion
        ) AS grp
    FROM connexions
)
SELECT
    user_id,
    MIN(date_connexion) AS debut_serie,
    MAX(date_connexion) AS fin_serie,
    COUNT(*) AS jours_consecutifs
FROM numbered
GROUP BY user_id, grp
ORDER BY user_id, debut_serie;
"""
print(sql_2_6_solution)

In [ ]:
# === SOLUTION 4.5 (Qualite des donnees) ===

def check_nulls(df, colonnes):
    """Retourne les colonnes ayant des NaN et leur pourcentage."""
    null_info = df[colonnes].isnull().sum()
    null_pct = (null_info / len(df) * 100).round(2)
    result = pd.DataFrame({'nb_nulls': null_info, 'pct_nulls': null_pct})
    return result[result['nb_nulls'] > 0]

def check_duplicates(df, cles):
    """Retourne les doublons sur les colonnes cles."""
    mask = df.duplicated(subset=cles, keep=False)
    return df[mask]

def check_range(df, colonne, min_val, max_val):
    """Retourne les lignes hors bornes."""
    return df[(df[colonne] < min_val) | (df[colonne] > max_val)]

def check_referential_integrity(df_fact, df_dim, fk, pk):
    """Verifie que toutes les FK de la table de faits existent dans la dimension."""
    orphans = df_fact[~df_fact[fk].isin(df_dim[pk])]
    return orphans

# Test
print("Fonctions de validation definies.")